In [1]:
import dask
import dask.distributed
from dask_util import DaskClient
import numpy as np

In [2]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

client = DaskClient(local_params=local_params)

2023-01-27 18:33:53,757 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-8ja517lo', purging
2023-01-27 18:33:53,758 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-k9bby83t', purging


In [3]:
client.getWorkerIds()

['tcp://127.0.0.1:33975',
 'tcp://127.0.0.1:36155',
 'tcp://127.0.0.1:39573',
 'tcp://127.0.0.1:44429']

In [4]:
%load_ext autoreload
%autoreload 2
import SepVector
import __pyDaskVector as DaskVector
import Hypercube
import pyOperator as Op


WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


In [5]:

ns = [10,10]
os = [0,0]
ds = [1,1]
chunks = (1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os, storage='dataFloat')
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=10	o=0.000000	d=1.000000

In [6]:
# option 1
# creating from scratch
data = DaskVector.DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [7]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector.DaskVector(client, from_vector=vec, chunks=chunks)

In [8]:
client.getClient().has_what()

{'tcp://127.0.0.1:33975': ('floatVector-a023098c3899e422851925c2c2e4c418',),
 'tcp://127.0.0.1:36155': ('floatVector-707131badb0aeab0f2f4850fd0a95697',
  'window-30a290d97e7a9ebba72b52adca09c7b1'),
 'tcp://127.0.0.1:39573': ('floatVector-0c076393fcc094234abd4294089ce986',
  'window-6a008b656e984a987b7fa9c39f9f8517'),
 'tcp://127.0.0.1:44429': ('window-cdfb10acb5a29b4717e6391eecff877d',)}

In [9]:
# daskVec[3:5,:] = 0

In [10]:
daskVec[:]

[array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32),
 array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32),
 array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]], dtype=float32)]

In [11]:

# zeroOp = DaskVector.DaskOperator(client, opCls=Op.ZeroOp, domain=daskVec, range=daskVec)

In [12]:
# zeroOp.forward(False, daskVec, daskVec)

In [13]:
scaleOp = DaskVector.DaskOperator(client, opCls=Op.scalingOp, domain=daskVec, range=data, op_args=[4])

In [14]:

# for _ in range(10):
#     scaleOp.forward(False, daskVec, data)
#     err = []
#     for v1, v2 in zip(data[:], daskVec[:]):
#         err.append(np.linalg.norm(v1-4*v2))
#     print(err)

In [15]:
data[:]

[array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)]

In [25]:
scaleOp.forward(False, daskVec, data)

In [26]:
data[:]

[array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32),
 array([[4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.],
        [4., 4., 4., 4., 4., 4., 4., 4., 4., 4.]], dtype=float32)]

In [27]:
scaleOp.adjoint(False, daskVec, data)

In [31]:
daskVec[:]

[array([[16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.]], dtype=float32),
 array([[16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16.]], dtype=float32)]